# PySeis Gain & AGC Demo
This notebook demonstrates how to replicate Seismic Unix processing pipelines (like `sugain` and `suxwigb` clipping) using PySeis.

In [ ]:
%pip install bokeh segyio

In [ ]:
# Install pyseiskit from PyPI
%pip install pyseiskit
# Install for local development
# %pip install -e ..

## 1A. Option 1: Create Simple Synthetic Data
Run this cell to generate a small 2D array representing 5 seismic traces, each with 100 time samples.

In [ ]:
import numpy as np

# 100 samples, 5 traces
data = np.random.randn(100, 5) 
times = np.arange(100)
offsets = np.arange(5, dtype=float)


## 1B. Option 2: Load Real `.su` or `.sgy` Data
Run this cell instead if you have the file `tac-204RL239.su` in the same folder as this notebook.

In [ ]:
import segyio
from seismic_reader import read_seismic_file

filename = 'tac-204RL239.su'

# We only want to plot a single gather to avoid rendering thousands of traces.
# FieldRecord corresponds to the 'fldr' header in SU/SEGY.
gather_key = segyio.TraceField.FieldRecord
gather_index = 50  # Change this to a valid fldr number from your dataset!

try:
    data, offsets, times = read_seismic_file(filename, gather_key=gather_key, gather_index=gather_index)
    print(f"Loaded {data.shape[1]} traces for fldr {gather_index} with {data.shape[0]} samples each.")
except FileNotFoundError:
    print(f"File '{filename}' not found. Please place it in the 'demos' folder or stick to Option 1A.")
except ValueError as e:
    print(f"Error: {e}")

## 2. Process Data for Wiggles
Regardless of how you loaded the data above, we now pass it through PySeis.

In [ ]:
from pyseiskit import sourceData
from pyseiskit import gain
from pyseiskit import clip

# Replicating SU pipeline: suwind | sugain agc=1 wagc=1.0 | suxwigb perc=99
dt = times[1] - times[0]

# 1. Apply AGC (Optional, just like sugain agc=1 wagc=1.0)
data_gained = gain.applyAGC(data, wagc=1.0, intervalTimeSamples=dt)
# data_gained = data  # uncomment above to apply AGC

# 2. Apply Percentile Clipping (Like suxwigb perc=99 flat-topping)
data_clipped = clip.applyPercentileClip(data_gained, percentile=99.0)

# 3. Scale physical dimensions (overlap=1.0 is SU default xcur/overlap)
scaled_data = sourceData.rescaleDataForWiggle(data_clipped, offsets, overlap=1.0)

# 4. Generate geometry
line_data = sourceData.wiggleLinesDataFactory(scaled_data, offsets, times)
patch_data = sourceData.wigglePatchesDataFactory(scaled_data, offsets, times, fill_mode='positive')

## 3. Visualize Results
Here we use Bokeh to visualize the gained results interactively.

In [ ]:
from bokeh.plotting import figure, show, output_notebook
output_notebook()

bokeh_plot = figure(width=600, height=400, y_range=(times[-1], times[0]))
bokeh_plot.multi_line(**line_data, color='black', line_width=0.5)
bokeh_plot.patches(**patch_data, color='black', line_width=0)

show(bokeh_plot)